# Data Science I - WS 2024/2025 - Praktikum IV: Modelling & Validation - California Housing Dataset


**Regeln für das Praktikum:**
- Sie können die Aufgaben alleine oder zu zweit bearbeiten.
- Sie dürfen in Vorlesungs-Skripten nachlesen
- Sollten Fragen auftauchen, oder die Verwendung von Klassen unklar sein, schlagen Sie  die Dokumentation online nach
- Befolgen Sie einfach den Anweisungen im Skript; manchmal wird im Markdown Text eine Frage gestellt; tragen Sie Ihre Antwort in der nächsten Markdown-Zelle bei #TODO ein!

**Hinweis:**

Wie in der Vorlesung verwenden wir für Modelling & Validation in diesem Praktikum die Pakete

* [pandas](https://pandas.pydata.org/docs/reference/)
* [Scikit Learn API](https://scikit-learn.org/stable/)

Sie finden alle Befehle, die Sie benötigen in den Vorlesungsaufzeichnunen.

In diesen Praktikum arbeiten wir mit der **California Housing Dataset**: Es enthält die mittleren Preise für Häuser in Bezirken von Californien zusammen mit den geografischen Daten wie Längen- und Breitengrad und Nähe zum Meer, Zensus-Daten wie den Einwohnerzahlen in jedem Bezirk, der Gesamtanzahl von Räumen und Haushalten, sowie dem mittleren Einkommen in den Bezirken. 

**Langfristiges Ziel:** Wir wollen mit Hilfe dieser Daten herausfinden, wie man den mittleren Hauspreis von den anderen gegebenen Daten ableiten kann.

**Ziel dieses Praktikums:** Wir wollen ML-Modelle für das gegebene Regressionsproblem  trainieren und diese Validieren 

Als Erstes öffen wir die Daten, die wir im letzten Praktikum bereits heruntergeladen haben. Übergeben Sie hier der Funktion `pd.read_csv()`den Pfad, wo Sie die "housing.csv" Datei gespeichert haben. Lassen Sie sich zur Kontrolle wieder die ersten 5 Zeilen des Datensatzes ausgeben.

In [80]:
import pandas as pd
import numpy as np

# laden Sie den Datensatz "housing.csv"
df = pd.read_csv('/home/farouk/Uni/Data Science/Praktika/datasets/housing/housing.csv')

In [81]:
# lassen Sie sich die ersten 5 Zeilen ausgeben
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


 ## 1. Feature Extraction
Fügen Sie die neuen Features "rooms_per_household", "bedrooms_per_room", "population_per_household" hinzu indem Sie gegebene Features kombinieren

In [82]:
#Definieren Sie die neuen Features als neue Spalten des Dataframes housing.

df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,6.281853,0.172096,2.181467


## 2. Train-Test Split 

**Wichtig:** Das muss immer vor der Bereininung der Daten stattfinden.

Nutzen Sie in der nächsten Zelle die Scikit-Learn Methode [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), um die Daten in train_set (80%) und test_set (20%) aufzuteilen.



In [83]:
#Verwenden Sie Sklearn's train_test_split für den Split in train_set und test_set
import sklearn
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, random_state=0, train_size=0.8)
#Lassen Sie sich die ersten Zeilen von test_set ausgeben. 

train.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,rooms_per_household,bedrooms_per_room,population_per_household
12069,-117.55,33.83,6.0,502.0,76.0,228.0,65.0,4.2386,500001.0,INLAND,7.723077,0.151394,3.507692
15925,-122.44,37.73,52.0,2381.0,492.0,1485.0,447.0,4.3898,270000.0,NEAR BAY,5.326622,0.206636,3.322148
11162,-118.00,33.83,26.0,1718.0,385.0,1022.0,368.0,3.9333,196100.0,<1H OCEAN,4.668478,0.224098,2.777174
4904,-118.26,34.01,38.0,697.0,208.0,749.0,206.0,1.4653,118800.0,<1H OCEAN,3.383495,0.298422,3.635922
4683,-118.36,34.08,52.0,2373.0,601.0,1135.0,576.0,3.1765,225000.0,<1H OCEAN,4.119792,0.253266,1.970486


Wie könnten Sie den Datensatz in 3 Mengen (Train, Validation & Test) aufteilen?

#TODO

Da die obige Methode rein zufällig, die Daten in die zwei Mengen aufteilt, kann dies zu nicht repräsentativen Trainingsdaten führen. Angenommen Sie trainieren ein Modell für ein Klassifikationsproblem (zB. Bilderekennung der Zahlen 0-9) und eine Klasse (zB. die Ziffer 9) kommt gar nicht in den Trainingdaten vor. Dann hat Ihr Modell natürlich keine Chance diese Klasse in den Testdaten richtig zu klassifizieren. 

Das kann aber nicht nur bei der Outputvariable ein Problem sein, auch eine kategorielle Inputvariable kann so schlecht in Trainingsdaten und Testdaten aufgeteilt sein, so dass wir nicht repräsentativen Trainingsdaten haben. Stellem Sie sich vor, alle wohlhabenden Bezirke sind in den Trainingsdaten gelandet und alle ärmeren Bezirke in den Testdaten, dann wird unser Modell ziemlich sicher falsche Prognosen auf den Testdaten generieren. 


Wir haben in der Vorlesung eine Methode, **Stratified Split**, gesehen um dies zu vermeiden. Gehen Sie wie in den Zellen beschrieben vor:

In [84]:
# Hinzufügen einer neuen Feature-Spalte genannt "income_cat", 
# welche jede Instance (also jeden District) je nach Mittlerem Einkommen
# in eine von 5 Einkommensklassen einteilt
# Vervollständigen Sie hierfür den Code und 

df["income_cat"] = pd.cut(df["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

df["income_cat"]

0        5
1        5
2        5
3        4
4        3
        ..
20635    2
20636    2
20637    2
20638    2
20639    2
Name: income_cat, Length: 20640, dtype: category
Categories (5, int64): [1 < 2 < 3 < 4 < 5]

In [85]:
# Wie viel Prozent der Instances sind in welcher Einkommensklasse? 
df["income_cat"].value_counts(normalize=True)

income_cat
3    0.350581
2    0.318847
4    0.176308
5    0.114438
1    0.039826
Name: proportion, dtype: float64

In [86]:
# modifizierter train_test_split Code mit stratify
train_set_strat, test_set_strat = train_test_split(df, random_state=0, train_size=0.8, stratify=df["income_cat"])

In [87]:
#Berechnen Sie wieder die prozentuale Verteilung der 5 Einkommensklassen 
#für train_set und test_set und vergleichen Sie diese mit dem Split ohne "stratify"
print(train_set_strat["income_cat"].value_counts(normalize=True))
print(test_set_strat["income_cat"].value_counts(normalize=True))

print(train_set_strat["income_cat"].value_counts(normalize=True))
print(test_set_strat["income_cat"].value_counts(normalize=True))

income_cat
3    0.350594
2    0.318859
4    0.176296
5    0.114462
1    0.039789
Name: proportion, dtype: float64
income_cat
3    0.350533
2    0.318798
4    0.176357
5    0.114341
1    0.039971
Name: proportion, dtype: float64
income_cat
3    0.350594
2    0.318859
4    0.176296
5    0.114462
1    0.039789
Name: proportion, dtype: float64
income_cat
3    0.350533
2    0.318798
4    0.176357
5    0.114341
1    0.039971
Name: proportion, dtype: float64


In [88]:
#Die 'income_cat' Spalte wird nun nicht mehr benötigt. Löschen Sie die Spalte in allen DataFrames. (Tipp: .drop() )
for df in [df, test_set_strat, train_set_strat]:
    df.drop("income_cat", axis=1, inplace=True, errors="ignore")
    print(df.head())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  \
0       322.0       126.0         8.3252            452600.0        NEAR BAY   
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY   
2       496.0       177.0         7.2574            352100.0        NEAR BAY   
3       558.0       219.0         5.6431            341300.0        NEAR BAY   
4       565.0       259.0         3.8462            342200.0        NEAR BAY   

   rooms_per_household  bedrooms_per_room  population_per_household 

## 3. Input-Label Split
Bevor wir mit dem Data Preprocessing fortfahren, trenne wir die Labels von den Input-Daten, das heißt wir teilen in $X$ und $y$ auf.

In [89]:
# Löschen Sie hierzu mit .drop die label-Spalte "median_house_value" und 
# nennen Sie den neuen DataFrame train 
# Kopieren Sie außerdem die Spalte median_house_value in einen DataFrame namens train_labels.
# Führen Sie dieselben Schritte durch für strat_test_set und nennen Sie die neuen DataFrames test und test_labels.
train = train_set_strat.drop(columns=["median_house_value"])
train_labels = train_set_strat["median_house_value"]
test = test_set_strat.drop(columns=["median_house_value"])
test_labels = test_set_strat["median_house_value"]

Speichern Sie die train_labels als csv-Datei "train_labels.csv" und die test_labels als csv-Datei "test_labels.csv" neben "housing.cvs" ab.

Setzen Sie dabei immer index = False!!! Warum ist das wichtig, falls Sie später die Daten einlesen und zum Trainieren eines Modells verwenden wollen? 

Antwort: Sonst wird die Index-Spalte als zusätzliche Datenspalte eingelesen.

In [90]:
# Speichern Sie die train_labels als csv-Datei "train_labels.csv" und die test_labels als csv-Datei "test_labels.csv" 
# neben "housing.cvs" ab indem Sie `.to_csv()` nutzen
train_labels.to_csv('/home/farouk/Uni/Data Science/Praktika/datasets/train_labels.csv', index=False)
test_labels.to_csv('/home/farouk/Uni/Data Science/Praktika/datasets/test_labels.csv', index=False)

## 4. Imputing Missing Values, Scaling Quantative Features, Encoding Categorical Variables

Teilen Sie die Features in quantitativ und kategoriell ein und übernehmen Sie den in Praktikum 3 Schritt 4 definierten Transformer  (für Imputing Missing Values, Scaling Quantative Features und Encoding Categorical Variables).

In [91]:
# teilen sie zunächst die Feature Name in kategoriell und quantitative ein. 
quant = list(df.drop(columns=["ocean_proximity",'median_house_value']).columns)  
cat = ["ocean_proximity"]  


In [92]:
# übernehmen Sie die Definition des Transformers aus Praktikum 3 Schritt 4
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

quant_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('std_scaler', StandardScaler()),
    ])

cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="most_frequent")),
        ('encoder', OneHotEncoder()),
    ])

full_pipeline = ColumnTransformer([
  ("quant", quant_pipeline, quant), 
  ("cat", cat_pipeline, cat),
])

In [93]:
# fitten und wenden Sie nun den Transformer auf train an (`.fit_transform()')
train = full_pipeline.fit_transform(train)

# wenden Sie nun den Transformer auf test an (der Transformer darf nicht nochmal auf tets gefitted werden, da zB. Medien zum Auffüllen
#immer auf der Trainingsmenge bestimmt werden müssen (`.transform()')
test = full_pipeline.transform(test)

In [94]:
# Wandeln Sie X_train und X_test wieder in einen DataFrame um

# Um alle Spaltennamen zu bekommen können Sie zB. folgendes nutzen
#cats_1hot = full_pipeline['cat']['encoder'].get_feature_names_out(['ocean_proximity'])
#all_attribs = num_attribs + list(cats_1hot)
cats_1hot = full_pipeline['cat']['encoder'].pet_feature_names


In [95]:
# Speichern Sie die beiden resultierenden Dataframes als als csv-Datei "housing_train.csv" bzw. "housing_test.csv"
# unter dem Pfad HOUSING_PATH ab! Setzen Sie dabei index = False wie zuvor (aus den gleichen Gründen!)



## 5. Modelling

Wir werden nun eine Lineare Regression auf den Daten trainieren, dazu nutzen wir Scikit-Learn und gehen vor wie in der Vorlesung.


In [96]:
# 1. Importieren Sie zuerst die Modelklasse Lineare Regression von Scikit-Learn.
from sklearn.linear_model import LinearRegression

In [97]:
# 2. Setzen Sie den Hyperparameter fit_intercept auf True
LinearRegression(fit_intercept=True)

LinearRegression()

In [98]:
# 3. Prüfen Sie ob die Feature Matrix X_train (und X_test) und der target vector train_label (und test_target) das passende Format haben


In [99]:
# 4. Training: fitten Sie das definierte Modell auf den Trainingsdaten


## 6. Hyperparameter Tuning & Validation

Definieren Sie ein zweites Model der [K-Nearest Neighbor Regression](https://scikit-learn.org/dev/auto_examples/neighbors/plot_regression.html) mit dem Hyperparameter `n_neighbors`. Sei zB. `n_neighbors=3`:
Ein 3-Nearest Neighbors Regression Modell ermittelt zu einem gegebenen Datenpunkt $x_\mathrm{new}$ die 3 nächsten Nachbarn $x_1,x_2,x_3$ in der Featurematrix und interpoliert dann die Vorhersage $f(x_\mathrm{new})$ aus den Werten $f(x_1),f(x_2),f(x_3)$. 
Wir werden dieses Modell in der Vorlesung noch genauer kennenlernen. 

Nutzen Sie [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) aus sklearn um herauszufinden welches K aus $[1,3,5,10,15]$ das beste K-Nearest Neighbors Regression Modell liefert.

Vergleichen Sie anschließend mit [cross_validation](https://scikit-learn.org/1.5/modules/cross_validation.htmlhttps://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) das Lineare Regressionsmodell mit dem Besten K-NN Modell.

Evaluieren Sie zuletzt den Gewinner auf den Testdaten.


In [100]:
# Finden Sie mit GridSearch wie in der VL den besten Hyperparameter K
# grid_search = GridSearchCV(__)
# nutzen Sie als Parameter {'n_neighbors': [1,3,5,10,15]}
# und scoring="r2"
# lassen Sie sich mit return_train_score=True den Trainscore mitberechnen
# lassen Sie sich die Ergebnisse mit pd.DataFrame(grid_search.cv_results_) ausgeben


Lesen Sie aus dem Ergebnisdataframe ab, ob es bei höherem oder niedrigerem K zu Overfitting kommt!

##TODO: 

In [101]:
# Vergleichen Sie die das KNN Gewinnermodell mit 
# dem Linearen Regressionsmodell durch cross_validation 
# Wählen Sie als Metrik den r^2 mit `scoring="r2"`


Was sagt der $R^2$-Score aus? Welches Modell performed besser?

##TODO

In [102]:
# Evaluieren Sie das bessere Modell auf den zurückgehaltenen Testdaten, indem Sie den R^2 Score bestimmen
